In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from keras.models import Sequential
from keras.layers import Dense, LSTM
from sklearn import metrics
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from scipy.stats import ttest_1samp

2025-01-25 12:45:14.260977: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-25 12:45:14.457169: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1737787514.591590    3758 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1737787514.643984    3758 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-25 12:45:14.948422: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
df=pd.read_csv("bitcoin.csv")
df.head()

,Date,Open,High,Low,Close,Volume,Month,Day,Year,Daily_Return,7-day MA,30-day MA,30-day Volatility
0,2015-01-01,320.434998,320.434998,314.002991,314.248993,8036550,1,1,2015,NaN,NaN,NaN,NaN
1,2015-01-02,314.079010,315.838989,313.565002,315.032013,7860650,1,2,2015,0.002492,NaN,NaN,NaN
2,2015-01-03,314.846008,315.149994,281.082001,281.082001,33054400,1,3,2015,-0.107767,NaN,NaN,NaN
3,2015-01-04,281.145996,287.230011,257.612000,264.195007,55629100,1,4,2015,-0.060079,NaN,NaN,NaN
4,2015-01-05,265.084015,278.341003,265.084015,274.473999,43962800,1,5,2015,0.038907,NaN,NaN,NaN


In [3]:
df.shape

(3560, 13)

In [4]:
df=df.dropna()

In [5]:
df.shape

(3530, 13)

In [6]:
df.tail()

,Date,Open,High,Low,Close,Volume,Month,Day,Year,Daily_Return,7-day MA,30-day MA,30-day Volatility
3555,2024-09-25,64302.589844,64804.503906,62945.375000,63143.144531,25078377700,9,25,2024,-0.018022,63421.699777,59364.688411,0.023105
3556,2024-09-26,63138.546875,65790.796875,62669.269531,65181.019531,36873129847,9,26,2024,0.032274,63741.780134,59553.917969,0.021436
3557,2024-09-27,65180.664062,66480.695312,64852.992188,65790.664062,32058813449,9,27,2024,0.009353,64112.878348,59779.352604,0.021355
3558,2024-09-28,65792.179688,66255.531250,65458.035156,65887.648438,15243637984,9,28,2024,0.001474,64468.993862,59996.001563,0.021355
3559,2024-09-29,65888.898438,66069.343750,65450.015625,65635.304688,14788214575,9,29,2024,-0.003830,64752.792969,60213.195833,0.021346


In [7]:
X=df.drop(['Close','Date'],axis=1)
y=df['Close']

In [8]:
# splitting X and y into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

In [9]:
# import the regressor
from sklearn.ensemble import RandomForestRegressor
  
 # create regressor object
model = RandomForestRegressor(n_estimators = 100, random_state = 0)
  
# fit the regressor with x and y data
model.fit(X_train, y_train) 
y_pred = model.predict(X_test)

In [10]:
print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))
print('r2_score:',r2_score(y_test,y_pred))

Mean Absolute Error: 172.40483090548665
Mean Squared Error: 140582.07973355666
Root Mean Squared Error: 374.942768610833
r2_score: 0.9996673365045837


In [11]:
# Metrics calculation
print('Mean Absolute Error (MAE):', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error (MSE):', mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error (RMSE):', np.sqrt(mean_squared_error(y_test, y_pred)))

# Mean Absolute Percentage Error (MAPE)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
print('Mean Absolute Percentage Error (MAPE):', mape)

# R-squared score
print('R-squared (R²):', r2_score(y_test, y_pred))

Mean Absolute Error (MAE): 172.40483090548665
Mean Squared Error (MSE): 140582.07973355666
Root Mean Squared Error (RMSE): 374.942768610833
Mean Absolute Percentage Error (MAPE): 1.0446096940616003
R-squared (R²): 0.9996673365045837


In [12]:
import scipy.stats as stats

# Hypothesis Test
# Null Hypothesis (H0): The mean difference between predicted and actual values is 0 (no difference).
# Alternative Hypothesis (H1): The mean difference between predicted and actual values is not 0 (significant difference).

# Calculate the difference between predicted and actual values
difference = y_test - y_pred.flatten()

# Perform a paired t-test
t_stat, p_value = stats.ttest_1samp(difference, 0)

# Display the t-statistic and p-value
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

# Step 7: Conclusion of Hypothesis Test
if p_value < 0.05:
    print("Reject the null hypothesis: There is a significant difference between predicted and actual values.")
else:
    print("Fail to reject the null hypothesis: No significant difference between predicted and actual values.")

T-statistic: -0.9295688233216983
P-value: 0.35284871211326696
Fail to reject the null hypothesis: No significant difference between predicted and actual values.


In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predictions on training data
y_train_pred = model.predict(X_train)

# Predictions on test data
y_test_pred = model.predict(X_test)

# Training data metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred)

# Testing data metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

# Print results
print("Training Data Metrics:")
print(f"Mean Absolute Error (MAE): {train_mae:.2f}")
print(f"Mean Squared Error (MSE): {train_mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {train_rmse:.2f}")
print(f"R-squared (R²): {train_r2:.4f}")

print("\nTesting Data Metrics:")
print(f"Mean Absolute Error (MAE): {test_mae:.2f}")
print(f"Mean Squared Error (MSE): {test_mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {test_rmse:.2f}")
print(f"R-squared (R²): {test_r2:.4f}")


Training Data Metrics:
Mean Absolute Error (MAE): 67.08
Mean Squared Error (MSE): 18189.26
Root Mean Squared Error (RMSE): 134.87
R-squared (R²): 1.0000

Testing Data Metrics:
Mean Absolute Error (MAE): 172.40
Mean Squared Error (MSE): 140582.08
Root Mean Squared Error (RMSE): 374.94
R-squared (R²): 0.9997


In [19]:
# Import the XGBoost regressor
from xgboost import XGBRegressor

# Create the XGBoost regressor object
xgb_model = XGBRegressor(n_estimators=100, random_state=0, learning_rate=0.1, max_depth=6)

# Fit the regressor with X_train and y_train data
xgb_model.fit(X_train, y_train)

# Predict using the XGBoost model
y_pred = xgb_model.predict(X_test)


In [22]:


# Training data metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred)

# Testing data metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

# Print the metrics
print("Training Data Metrics:")
print(f"Mean Absolute Error (MAE): {train_mae:.2f}")
print(f"Mean Squared Error (MSE): {train_mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {train_rmse:.2f}")
print(f"R-squared (R²): {train_r2:.4f}")

print("\nTesting Data Metrics:")
print(f"Mean Absolute Error (MAE): {test_mae:.2f}")
print(f"Mean Squared Error (MSE): {test_mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {test_rmse:.2f}")
print(f"R-squared (R²): {test_r2:.4f}")

Training Data Metrics:
Mean Absolute Error (MAE): 67.08
Mean Squared Error (MSE): 18189.26
Root Mean Squared Error (RMSE): 134.87
R-squared (R²): 1.0000

Testing Data Metrics:
Mean Absolute Error (MAE): 172.40
Mean Squared Error (MSE): 140582.08
Root Mean Squared Error (RMSE): 374.94
R-squared (R²): 0.9997
